In [2]:
import pandas as pd
import pandas_gbq
from google.cloud import bigquery
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

print("SIA Wevengers 분석 환경 준비 완료")

SIA Wevengers 분석 환경 준비 완료


In [3]:
# 1. 고위험군 CAMEO 코드 리스트
target_cameo_codes = [ 
    '150', '151', '152', '153', '154', '155', 
    '190', '191', '192', '193', '194', '195', '196', 
    '200', '201', '202', '203', '204'
]
formatted_codes = ", ".join([f"'{code}'" for code in target_cameo_codes])

query = f"""
SELECT 
    SQLDATE, 
    IsRootEvent,
    EventCode, 
    GoldsteinScale, 
    NumMentions,
    AvgTone,
    ActionGeo_Type,
    ActionGeo_Lat, 
    ActionGeo_Long, 
    SOURCEURL
FROM `gdelt-bq.full.events`
WHERE SQLDATE >= 20130401
  AND IsRootEvent = 1
  AND ActionGeo_Type IN (3, 4, 5)
  AND (
    (Actor1CountryCode = 'CHN' AND Actor2CountryCode = 'TWN') OR 
    (Actor1CountryCode = 'TWN' AND Actor2CountryCode = 'CHN')
  )
  AND EventCode IN ({formatted_codes})
"""

project_id = "project-083a4a68-3ded-4079-8bb" # 로그인창 연결됨
df = pandas_gbq.read_gbq(query, project_id=project_id)

# 결과 확인
print(f"2013년부터 2026년 현재까지 총 {len(df)}건의 데이터를 불러왔습니다.")
display(df.head())

Downloading: 100%|██████████|
2013년부터 2026년 현재까지 총 11897건의 데이터를 불러왔습니다.


,SQLDATE,IsRootEvent,EventCode,GoldsteinScale,NumMentions,AvgTone,ActionGeo_Type,ActionGeo_Lat,ActionGeo_Long,SOURCEURL
0,20160609,1,192,-9.5,2,-1.587302,4,29.0000,125.000,http://www.eyeontaiwan.com/sporadic-intense-sh...
1,20160609,1,192,-9.5,2,-3.786192,4,23.2783,120.314,http://focustaiwan.tw/news/asoc/201606090016.aspx
2,20160917,1,192,-9.5,10,-6.077348,5,26.5450,117.843,http://www.crcconnection.com/2016/09/17/watch-...
3,20161006,1,193,-10.0,6,-6.862745,4,22.1094,120.874,http://alert5.com/2016/10/06/taiwan-aborted-at...
4,20160701,1,153,-7.2,1,0.334672,4,25.0478,121.532,http://www.nippon.com/en/column/g00372/


In [4]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [5]:
# 판다스 출력 옵션 설정: 컬럼 너비 제한 해제
pd.set_option('display.max_colwidth', None)

# 만약 행(row) 개수도 더 많이 보고 싶다면 아래 설정도 유용합니다.
pd.set_option('display.max_rows', 100)

In [6]:
import pandas as pd
import os

# 1. 폴더 및 파일 경로 설정
folder_path = r'C:\Users\rudak\OneDrive\Desktop\SIA\DS8_SIA_Project'
file_name = 'gps.xlsx'
full_path = os.path.join(folder_path, file_name)

# 2. gps.xlsx 파일 존재 여부 확인 및 로드
if os.path.exists(full_path):
    gps_data = pd.read_excel(full_path)
    print(f"✅ 파일을 성공적으로 불러왔습니다: {full_path}")
    
    # 3. 매칭 준비: 소수점 정밀도 통일 (3자리 반올림)
    # GDELT 데이터셋 전처리
    gdelt_check = df.copy()
    gdelt_check['lat_match'] = gdelt_check['ActionGeo_Lat'].round(3)
    gdelt_check['long_match'] = gdelt_check['ActionGeo_Long'].round(3)
    
    # gps.xlsx 데이터 전처리 (컬럼명: 위도(Lat), 경도(Long))
    gps_data['lat_match'] = gps_data['위도(Lat)'].round(3)
    gps_data['long_match'] = gps_data['경도(Long)'].round(3)
    
    # 4. 데이터 병합(Merge)을 통한 포함 여부 확인
    matched_df = pd.merge(
        gdelt_check,
        gps_data[['lat_match', 'long_match']],
        on=['lat_match', 'long_match'],
        how='inner'
    )
    
    # 5. 결과 보고
    total_gps_points = len(gps_data.drop_duplicates(['lat_match', 'long_match']))
    found_articles = len(matched_df)
    
    print("-" * 50)
    print(f"📍 검색한 고유 좌표 수: {total_gps_points}개")
    print(f"📰 GDELT 내 매칭된 기사 수: {found_articles}건")
    print("-" * 50)
    
    if found_articles > 0:
        print("최근 매칭 기사 샘플 (상위 10건):")
        display(matched_df[['SQLDATE', 'EventCode', 'ActionGeo_Lat', 'ActionGeo_Long', 'SOURCEURL']].head(10))
    else:
        print("⚠️ 일치하는 기사가 없습니다. 다음을 확인해 보세요:")
        print("1. GDELT 데이터의 기간이 gps.xlsx의 사건 시점과 일치하는가?")
        print("2. 반올림 자릿수를 .round(2)로 낮춰서 다시 시도해 보세요.")
else:
    print(f"❌ 파일을 찾을 수 없습니다: {full_path}")
    print("경로가 정확한지, 혹은 파일명이 'gps.xlsx'가 맞는지 다시 한번 확인해 주세요.")

✅ 파일을 성공적으로 불러왔습니다: C:\Users\rudak\OneDrive\Desktop\SIA\DS8_SIA_Project\gps.xlsx
--------------------------------------------------
📍 검색한 고유 좌표 수: 68개
📰 GDELT 내 매칭된 기사 수: 0건
--------------------------------------------------
⚠️ 일치하는 기사가 없습니다. 다음을 확인해 보세요:
1. GDELT 데이터의 기간이 gps.xlsx의 사건 시점과 일치하는가?
2. 반올림 자릿수를 .round(2)로 낮춰서 다시 시도해 보세요.


In [9]:
import pandas as pd
import os

# 1. 폴더 및 파일 경로 설정
folder_path = r'C:\Users\rudak\OneDrive\Desktop\SIA\DS8_SIA_Project'
file_name = 'gps.xlsx'
full_path = os.path.join(folder_path, file_name)

# 2. gps.xlsx 파일 존재 여부 확인 및 로드
if os.path.exists(full_path):
    gps_data = pd.read_excel(full_path)
    print(f"✅ 파일을 성공적으로 불러왔습니다: {full_path}")
    
    # 3. 매칭 준비: 소수점 정밀도 통일 (3자리 반올림)
    # GDELT 데이터셋 전처리
    gdelt_check = df.copy()
    gdelt_check['lat_match'] = gdelt_check['ActionGeo_Lat'].round(2)
    gdelt_check['long_match'] = gdelt_check['ActionGeo_Long'].round(2)
    
    # gps.xlsx 데이터 전처리 (컬럼명: 위도(Lat), 경도(Long))
    gps_data['lat_match'] = gps_data['위도(Lat)'].round(2)
    gps_data['long_match'] = gps_data['경도(Long)'].round(2)
    
    # 4. 데이터 병합(Merge)을 통한 포함 여부 확인
    matched_df = pd.merge(
        gdelt_check,
        gps_data[['lat_match', 'long_match']],
        on=['lat_match', 'long_match'],
        how='inner'
    )
    
    # 5. 결과 보고
    total_gps_points = len(gps_data.drop_duplicates(['lat_match', 'long_match']))
    found_articles = len(matched_df)
    
    print("-" * 50)
    print(f"📍 검색한 고유 좌표 수: {total_gps_points}개")
    print(f"📰 GDELT 내 매칭된 기사 수: {found_articles}건")
    print("-" * 50)
    
    if found_articles > 0:
        print("최근 매칭 기사 샘플 (상위 10건):")
        display(matched_df[['SQLDATE', 'EventCode', 'ActionGeo_Lat', 'ActionGeo_Long', 'SOURCEURL']].head(30))
    else:
        print("⚠️ 일치하는 기사가 없습니다. 다음을 확인해 보세요:")
        print("1. GDELT 데이터의 기간이 gps.xlsx의 사건 시점과 일치하는가?")
        print("2. 반올림 자릿수를 .round(2)로 낮춰서 다시 시도해 보세요.")
else:
    print(f"❌ 파일을 찾을 수 없습니다: {full_path}")
    print("경로가 정확한지, 혹은 파일명이 'gps.xlsx'가 맞는지 다시 한번 확인해 주세요.")

✅ 파일을 성공적으로 불러왔습니다: C:\Users\rudak\OneDrive\Desktop\SIA\DS8_SIA_Project\gps.xlsx
--------------------------------------------------
📍 검색한 고유 좌표 수: 63개
📰 GDELT 내 매칭된 기사 수: 77건
--------------------------------------------------
최근 매칭 기사 샘플 (상위 10건):


,SQLDATE,EventCode,ActionGeo_Lat,ActionGeo_Long,SOURCEURL
0,20160701,194,23.5700,119.570,http://www.business-standard.com/article/international/taiwan-mistakenly-fires-supersonic-missile-towards-china-116070100308_1.html
1,20160701,194,23.5700,119.570,http://www.thehindu.com/news/international/taiwan-mistakenly-fires-supersonic-missile-towards-china/article8796433.ece
2,20160701,194,22.6886,120.293,http://www.thenational.ae/taiwan-mistakenly-fires-missile-towards-china-hitting-trawler
3,20160701,194,22.6886,120.293,http://www.thenational.ae/taiwan-mistakenly-fires-missile-towards-china-hitting-trawler
4,20160701,194,22.6886,120.293,http://www.thenational.ae/taiwan-mistakenly-fires-missile-towards-china-hitting-trawler
5,20160701,194,22.6886,120.293,http://www.thenational.ae/taiwan-mistakenly-fires-missile-towards-china-hitting-trawler
6,20160701,194,23.5700,119.570,http://www.thehindu.com/news/international/taiwan-mistakenly-fires-supersonic-missile-towards-china/article8796433.ece
7,20160701,194,23.5700,119.570,http://www.rediff.com/news/report/taiwan-fires-supersonic-missile-at-china-by-error/20160701.htm
8,20160702,194,22.6886,120.293,http://borneobulletin.com.bn/taiwan-mistakenly-fires-missile-towards-china/
9,20160702,194,22.6886,120.293,http://borneobulletin.com.bn/taiwan-mistakenly-fires-missile-towards-china/


In [8]:
import pandas as pd
import os

# 1. 폴더 및 파일 경로 설정
folder_path = r'C:\Users\rudak\OneDrive\Desktop\SIA\DS8_SIA_Project'
file_name = 'gps.xlsx'
full_path = os.path.join(folder_path, file_name)

# 2. gps.xlsx 파일 존재 여부 확인 및 로드
if os.path.exists(full_path):
    gps_data = pd.read_excel(full_path)
    print(f"✅ 파일을 성공적으로 불러왔습니다: {full_path}")
    
    # 3. 매칭 준비: 소수점 정밀도 통일 (3자리 반올림)
    # GDELT 데이터셋 전처리
    gdelt_check = df.copy()
    gdelt_check['lat_match'] = gdelt_check['ActionGeo_Lat'].round(1)
    gdelt_check['long_match'] = gdelt_check['ActionGeo_Long'].round(1)
    
    # gps.xlsx 데이터 전처리 (컬럼명: 위도(Lat), 경도(Long))
    gps_data['lat_match'] = gps_data['위도(Lat)'].round(1)
    gps_data['long_match'] = gps_data['경도(Long)'].round(1)
    
    # 4. 데이터 병합(Merge)을 통한 포함 여부 확인
    matched_df = pd.merge(
        gdelt_check,
        gps_data[['lat_match', 'long_match']],
        on=['lat_match', 'long_match'],
        how='inner'
    )
    
    # 5. 결과 보고
    total_gps_points = len(gps_data.drop_duplicates(['lat_match', 'long_match']))
    found_articles = len(matched_df)
    
    print("-" * 50)
    print(f"📍 검색한 고유 좌표 수: {total_gps_points}개")
    print(f"📰 GDELT 내 매칭된 기사 수: {found_articles}건")
    print("-" * 50)
    
    if found_articles > 0:
        print("최근 매칭 기사 샘플 (상위 10건):")
        display(matched_df[['SQLDATE', 'EventCode', 'ActionGeo_Lat', 'ActionGeo_Long', 'SOURCEURL']].head(30))
    else:
        print("⚠️ 일치하는 기사가 없습니다. 다음을 확인해 보세요:")
        print("1. GDELT 데이터의 기간이 gps.xlsx의 사건 시점과 일치하는가?")
        print("2. 반올림 자릿수를 .round(2)로 낮춰서 다시 시도해 보세요.")
else:
    print(f"❌ 파일을 찾을 수 없습니다: {full_path}")
    print("경로가 정확한지, 혹은 파일명이 'gps.xlsx'가 맞는지 다시 한번 확인해 주세요.")

✅ 파일을 성공적으로 불러왔습니다: C:\Users\rudak\OneDrive\Desktop\SIA\DS8_SIA_Project\gps.xlsx
--------------------------------------------------
📍 검색한 고유 좌표 수: 45개
📰 GDELT 내 매칭된 기사 수: 9162건
--------------------------------------------------
최근 매칭 기사 샘플 (상위 10건):


,SQLDATE,EventCode,ActionGeo_Lat,ActionGeo_Long,SOURCEURL
0,20160701,153,25.0478,121.532,http://www.nippon.com/en/column/g00372/
1,20160701,153,25.0478,121.532,http://www.nippon.com/en/column/g00372/
2,20160701,153,25.0478,121.532,http://www.nippon.com/en/column/g00372/
3,20160701,194,23.5700,119.570,http://www.business-standard.com/article/international/taiwan-mistakenly-fires-supersonic-missile-towards-china-116070100308_1.html
4,20160701,194,23.5700,119.570,http://www.business-standard.com/article/international/taiwan-mistakenly-fires-supersonic-missile-towards-china-116070100308_1.html
5,20160701,194,23.5700,119.570,http://www.thehindu.com/news/international/taiwan-mistakenly-fires-supersonic-missile-towards-china/article8796433.ece
6,20160701,194,23.5700,119.570,http://www.thehindu.com/news/international/taiwan-mistakenly-fires-supersonic-missile-towards-china/article8796433.ece
7,20160701,194,25.0478,121.532,http://tribune.com.pk/story/1134046/taiwan-mistakenly-fires-carrier-killer-missile-towards-china/
8,20160701,194,25.0478,121.532,http://tribune.com.pk/story/1134046/taiwan-mistakenly-fires-carrier-killer-missile-towards-china/
9,20160701,194,25.0478,121.532,http://tribune.com.pk/story/1134046/taiwan-mistakenly-fires-carrier-killer-missile-towards-china/
